# 00 — Schema explore

Read-only exploration of the prepared DB source once available.

1. Load `.env` / engine from `db.connection`
2. `INFORMATION_SCHEMA.COLUMNS` for the chosen table/view
3. Fill `fraud_guard.features.ColumnMapping`
4. Do **not** write to the database

In [1]:
## Шаг A — путь + подключение

In [23]:
from pathlib import Path
import sys
from sqlalchemy import text

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path[:0] = [str(ROOT), str(ROOT / "src")]

from db.config import get_settings
get_settings.cache_clear()

from db.connection import get_engine

engine = get_engine()
VIEW = "dbo.TargetsByMetrics_RateGetAnswers"

with engine.connect() as conn:
    db = conn.execute(text("SELECT DB_NAME()")).scalar()
    n = conn.execute(text(f"SELECT COUNT_BIG(*) FROM {VIEW}")).scalar()

print("DB:", db)
print("rows:", n)

DB: SummerCampain
rows: 1212609


In [3]:
## Шаг B — загрузить таблицу в pandas (без печати PII)

In [4]:
import pandas as pd

df = pd.read_sql(f"SELECT * FROM {VIEW}", engine)
print(df.shape)
print(df.columns.tolist())
df.dtypes

(1212609, 35)
['ParticipateNumber', 'AnswerDate', 'AnswerTime', 'DiscountTypeDesc', 'BlackList', 'PrintStore', 'PrintDate', 'PrintPos', 'PrintOrder', 'PrintDateTime', 'EmpIdNumber', 'EmpStore', 'EmpName', 'EmpLastName', 'EmpStartDate', 'EmpEndDate', 'EmpRole', 'Year', 'Month', 'PhoneFromLog', 'ContactType', 'UserName', 'UserContact', 'ext_user_id', 'OveringDone', 'Reprinted', 'PrintSource', 'TakeChanel', 'PayChanel', 'OrderChanel', 'GreenReceipt', 'IdentifiedCustomer', 'Question_ID', 'Title', 'Answer_Value']


ParticipateNumber                str
AnswerDate            datetime64[us]
AnswerTime            datetime64[us]
DiscountTypeDesc                 str
BlackList                        str
PrintStore                   float64
PrintDate             datetime64[us]
PrintPos                     float64
PrintOrder                   float64
PrintDateTime         datetime64[us]
EmpIdNumber                  float64
EmpStore                     float64
EmpName                          str
EmpLastName                      str
EmpStartDate                     str
EmpEndDate                       str
EmpRole                          str
Year                           int64
Month                          int64
PhoneFromLog                     str
ContactType                      str
UserName                         str
UserContact                      str
ext_user_id                    int64
OveringDone                      str
Reprinted                        str
PrintSource                      str
T

In [5]:
df[["PrintStore", "Year", "Month", "Question_ID", "Answer_Value", "PhoneFromLog"]].head()

,PrintStore,Year,Month,Question_ID,Answer_Value,PhoneFromLog
0,1.0,2025,10,NaN,NaN,NaN
1,1.0,2025,10,10012.0,5.0,NaN
2,1.0,2025,10,10012.0,3.0,NaN
3,1.0,2025,10,10012.0,5.0,NaN
4,1.0,2025,10,10012.0,4.0,NaN


In [6]:
## Шаг C — период и объём

In [7]:
print(df["AnswerTime"].min(), df["AnswerTime"].max())
print(df.groupby(["Year", "Month"]).size())

2025-01-01 01:03:38 2026-08-27 04:54:43
Year  Month
2025  1        56282
      2        51260
      3        60106
      4        63024
      5        60099
      6        46370
      7        73251
      8        96612
      9        75470
      10       74463
      11       66405
      12       59663
2026  1        53800
      2        50506
      3        55209
      4        55969
      5        64015
      6        59644
      7        51610
      8        38851
dtype: int64


In [8]:
print("stores:", df["PrintStore"].nunique())
print("PrintDateTime range:", df["PrintDateTime"].min(), "→", df["PrintDateTime"].max())

stores: 245
PrintDateTime range: 2025-01-01 00:25:17 → 2026-08-27 04:36:17


In [9]:
## добить ясность (опционально, до D)

In [10]:
print("unique ParticipateNumber:", df["ParticipateNumber"].nunique())
print("unique PrintStore+Pos+Order+PrintDateTime:", 
      df.groupby(["PrintStore","PrintPos","PrintOrder","PrintDateTime"]).ngroups)
print("rows per participation: min/median/max =",
      df.groupby("ParticipateNumber").size().agg(["min","median","max"]).to_dict())
print("AnswerDate nunique:", df["AnswerDate"].nunique())
print("AnswerDate value_counts:\n", df["AnswerDate"].value_counts())

unique ParticipateNumber: 1212582
unique PrintStore+Pos+Order+PrintDateTime: 1212067
rows per participation: min/median/max = {'min': 1.0, 'median': 1.0, 'max': 4.0}
AnswerDate nunique: 602
AnswerDate value_counts:
 AnswerDate
2025-08-28    4288
2025-08-07    3872
2025-10-09    3706
2025-08-21    3663
2025-08-05    3621
              ... 
2026-03-01     510
2025-06-15     302
2025-06-13      32
2026-02-28      15
2026-08-27      12
Name: count, Length: 602, dtype: int64


In [11]:
## точно убедиться

In [12]:
# 1) Календарные даты ответа
print("AnswerDate unique:", sorted(df["AnswerDate"].dropna().dt.normalize().unique()))
print("nunique AnswerDate:", df["AnswerDate"].dt.normalize().nunique())

# 2) Календарные даты времени ответа
print("AnswerTime dates unique:", sorted(df["AnswerTime"].dropna().dt.date.unique()))
print("nunique AnswerTime.date:", df["AnswerTime"].dt.date.nunique())

# 3) Бизнес-дата заказа (PrintDate)
print("PrintDate unique:", sorted(df["PrintDate"].dropna().dt.normalize().unique()))
print("nunique PrintDate:", df["PrintDate"].dt.normalize().nunique())

# 4) Year/Month
print(df.groupby(["Year", "Month"]).size())

# 5) Жёсткая проверка: всё ли в одну дату ответа?
d0 = df["AnswerTime"].dt.date.min()
print("all AnswerTime on same day?", (df["AnswerTime"].dt.date == d0).all())

AnswerDate unique: [Timestamp('2025-01-01 00:00:00'), Timestamp('2025-01-02 00:00:00'), Timestamp('2025-01-03 00:00:00'), Timestamp('2025-01-04 00:00:00'), Timestamp('2025-01-05 00:00:00'), Timestamp('2025-01-06 00:00:00'), Timestamp('2025-01-07 00:00:00'), Timestamp('2025-01-08 00:00:00'), Timestamp('2025-01-09 00:00:00'), Timestamp('2025-01-10 00:00:00'), Timestamp('2025-01-11 00:00:00'), Timestamp('2025-01-12 00:00:00'), Timestamp('2025-01-13 00:00:00'), Timestamp('2025-01-14 00:00:00'), Timestamp('2025-01-15 00:00:00'), Timestamp('2025-01-16 00:00:00'), Timestamp('2025-01-17 00:00:00'), Timestamp('2025-01-18 00:00:00'), Timestamp('2025-01-19 00:00:00'), Timestamp('2025-01-20 00:00:00'), Timestamp('2025-01-21 00:00:00'), Timestamp('2025-01-22 00:00:00'), Timestamp('2025-01-23 00:00:00'), Timestamp('2025-01-24 00:00:00'), Timestamp('2025-01-25 00:00:00'), Timestamp('2025-01-26 00:00:00'), Timestamp('2025-01-27 00:00:00'), Timestamp('2025-01-28 00:00:00'), Timestamp('2025-01-29 00:00:

In [13]:
## Шаг D — идентификаторы клиента (заполненность)

In [14]:
for col in ["UserContact", "PhoneFromLog", "ext_user_id", "ParticipateNumber"]:
    non_null = df[col].notna() & (df[col].astype(str).str.strip() != "") & (df[col].astype(str) != "None")
    print(col, "fill_rate=", round(non_null.mean(), 3), "nunique=", df.loc[non_null, col].nunique())

UserContact fill_rate= 0.432 nunique= 186759
PhoneFromLog fill_rate= 0.046 nunique= 12674
ext_user_id fill_rate= 1.0 nunique= 147953
ParticipateNumber fill_rate= 1.0 nunique= 1212582


In [ ]:
## быстрая проверка ext_user_id (важно):

In [24]:
print(df["ext_user_id"].value_counts().head(10))
print("share ext_user_id==0:", (df["ext_user_id"] == 0).mean().round(3))
print("nunique where ext_user_id!=0:", df.loc[df["ext_user_id"] != 0, "ext_user_id"].nunique())

ext_user_id
0          527654
49052         334
622457        292
424086        270
5326          236
1792          227
1501          196
7725          172
25228         168
1260159       166
Name: count, dtype: int64
share ext_user_id==0: 0.435
nunique where ext_user_id!=0: 147952


In [15]:
## Шаг E — вопросы и оценки

In [16]:
print(df.groupby(["Question_ID", "Title"])["Answer_Value"].agg(["count", "mean", "min", "max"]))
print(df["Answer_Value"].value_counts().sort_index())

                           count      mean  min  max
Question_ID Title                                   
10012.0     חווית הקנייה  958598  4.418765  1.0  5.0
Answer_Value
1.0     37682
2.0     27967
3.0     82310
4.0    157922
5.0    652717
Name: count, dtype: int64


In [ ]:
##дополнение

In [25]:
print(df.groupby(["Question_ID", "Title"], dropna=False)["Answer_Value"]
        .agg(count="count", mean="mean", min="min", max="max")
        .sort_values("count", ascending=False)
        .head(20))
print("Answer_Value value_counts:\n", df["Answer_Value"].value_counts(dropna=False).sort_index())

                           count      mean  min  max
Question_ID Title                                   
10012.0     חווית הקנייה  958598  4.418765  1.0  5.0
NaN         NaN                0       NaN  NaN  NaN
Answer_Value value_counts:
 Answer_Value
1.0     37682
2.0     27967
3.0     82310
4.0    157922
5.0    652717
NaN    254011
Name: count, dtype: int64


In [17]:
## Шаг F — latency (секунды от заказа до ответа)

In [18]:
lat = (pd.to_datetime(df["AnswerTime"]) - pd.to_datetime(df["PrintDateTime"])).dt.total_seconds()
print(lat.describe())

count    1.212094e+06
mean     2.162658e+03
std      1.011113e+04
min     -1.358000e+03
25%      1.242000e+03
50%      1.885000e+03
75%      2.855000e+03
max      1.104663e+07
dtype: float64


In [ ]:
## дополнение

In [26]:
ans = df[df["Question_ID"].eq(10012) & df["Answer_Value"].notna()].copy()
print("answered rows:", len(ans))
lat = (pd.to_datetime(ans["AnswerTime"]) - pd.to_datetime(ans["PrintDateTime"])).dt.total_seconds()
print(lat.describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
print("lat < 0 (answer before order):", (lat < 0).sum())
print("lat <= 60 sec:", (lat <= 60).sum(), "share:", round((lat <= 60).mean(), 3))
print("lat <= 300 sec:", (lat <= 300).sum(), "share:", round((lat <= 300).mean(), 3))

answered rows: 958598
count    958164.000000
mean       1830.008294
std        1021.085105
min         302.000000
50%        1617.000000
90%        3274.000000
95%        3835.000000
99%        4664.000000
max       85634.000000
dtype: float64
lat < 0 (answer before order): 0
lat <= 60 sec: 0 share: 0.0
lat <= 300 sec: 0 share: 0.0


In [19]:
## Шаг G — BlackList

In [28]:
print(df["BlackList"].value_counts(dropna=False))

BlackList
לא                1192334
עובד                17963
הפחתות בקופה         2288
הטבת עובד קבוע         24
Name: count, dtype: int64


In [ ]:
## дополнение

In [29]:
print(ans["BlackList"].value_counts(dropna=False))
# доля top-box среди blacklist vs нет (без печати контактов)
print(
    ans.assign(top=(ans["Answer_Value"] == 5))
       .groupby(ans["BlackList"].fillna("NA"), dropna=False)["top"]
       .agg(n="count", top_box_rate="mean")
)

BlackList
לא                938927
עובד               17361
הפחתות בקופה        2287
הטבת עובד קבוע        23
Name: count, dtype: int64
                     n  top_box_rate
BlackList                           
הטבת עובד קבוע      23      0.913043
הפחתות בקופה      2287      0.882816
לא              938927      0.675244
עובד             17361      0.960313


In [21]:
## Шаг H — baseline 5% (до фильтров) + черновик mapping

In [30]:
# только рабочий вопрос + ответ
base = ans[ans["BlackList"].eq("לא")].copy()
def five_percent(s):
    return 100.0 * (s == 5).sum() / len(s)
print("overall 5% (BlackList=לא):", round(five_percent(base["Answer_Value"]), 2))
print("overall 5% (all answered):", round(five_percent(ans["Answer_Value"]), 2))
# store x month panel
panel = (base.groupby(["PrintStore", "Year", "Month"])["Answer_Value"]
           .agg(volume="count", five_pct=five_percent)
           .reset_index())
print(panel["five_pct"].describe())
print("panel rows:", len(panel))

overall 5% (BlackList=לא): 67.52
overall 5% (all answered): 68.09
count    4646.000000
mean       67.107078
std        12.036135
min         0.000000
25%        58.472995
50%        66.551434
75%        75.058411
max       100.000000
Name: five_pct, dtype: float64
panel rows: 4646


In [ ]:
## И mapping:

In [31]:
from fraud_guard.features import ColumnMapping
mapping = ColumnMapping(
    entity_key="UserContact",  # + fallback later
    store_id="PrintStore",
    event_ts="AnswerTime",
    survey_id="ParticipateNumber",
    question_id="Question_ID",
    answer_value="Answer_Value",
    channel="ContactType",
    checkout_ts="PrintDateTime",
)
mapping

ColumnMapping(entity_key='UserContact', store_id='PrintStore', event_ts='AnswerTime', survey_id='ParticipateNumber', question_id='Question_ID', answer_value='Answer_Value', channel='ContactType', checkout_ts='PrintDateTime')